# Original HPDC-Net baseline
Reproducible evaluation of the trained checkpoint.

Run all cells to evaluate on the held-out tomato test set. This notebook does **not** train a model.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    average_precision_score,
    confusion_matrix,
)


## Original architecture definitions


In [ ]:
class DepthwiseAdaptiveResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(DepthwiseAdaptiveResidualBlock, self).__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=3, stride=stride, padding=1, groups=in_channels
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return F.relu(x + identity)

class FeatureRecalibrationAttention(nn.Module):
    def __init__(self, in_channels):
        super(FeatureRecalibrationAttention, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        attention = self.sigmoid(self.conv1(x))
        return x * attention

class HybridPoolingLayer(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super(HybridPoolingLayer, self).__init__()
        self.max_pool = nn.MaxPool2d(kernel_size, stride)
        self.avg_pool = nn.AvgPool2d(kernel_size, stride)

    def forward(self, x):
        max_pooled = self.max_pool(x)
        avg_pooled = self.avg_pool(x)
        return (max_pooled + avg_pooled) / 2

class ConditionalFeatureEnhancement(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ConditionalFeatureEnhancement, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)

    def forward(self, x):
        b, c, _, _ = x.size()
        weights = self.global_avg_pool(x).view(b, c)
        weights = F.relu(self.fc1(weights))
        weights = torch.sigmoid(self.fc2(weights)).view(b, c, 1, 1)
        return x * weights

class MultiScaleFusionBlock(nn.Module):
    def __init__(self, in_channels):
        super(MultiScaleFusionBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.conv3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2)

    def forward(self, x):
        x1 = self.conv1(x)
        x3 = self.conv3(x)
        x5 = self.conv5(x)
        return x1 + x3 + x5

class HPDCNet(nn.Module):
    def __init__(self, num_classes=10):
        super(HPDCNet, self).__init__()
        # Stage 1
        self.layer1 = DepthwiseAdaptiveResidualBlock(3, 16)
        self.attn1 = FeatureRecalibrationAttention(16)
        self.pool1 = HybridPoolingLayer()

        # Stage 2
        self.layer2 = DepthwiseAdaptiveResidualBlock(16, 32)
        self.attn2 = FeatureRecalibrationAttention(32)
        self.pool2 = HybridPoolingLayer()

        # Stage 3
        self.layer3 = DepthwiseAdaptiveResidualBlock(32, 64)
        self.fusion3 = MultiScaleFusionBlock(64)
        self.attn3 = FeatureRecalibrationAttention(64)
        self.pool3 = HybridPoolingLayer()

        # Stage 4
        self.layer4 = DepthwiseAdaptiveResidualBlock(64, 128)
        self.fusion4 = MultiScaleFusionBlock(128)
        self.attn4 = FeatureRecalibrationAttention(128)

        # Classifier
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # Stage 1
        x = self.layer1(x)
        x = self.attn1(x)
        x = self.pool1(x)

        # Stage 2
        x = self.layer2(x)
        x = self.attn2(x)
        x = self.pool2(x)

        # Stage 3
        x = self.layer3(x)
        x = self.fusion3(x)
        x = self.attn3(x)
        x = self.pool3(x)

        # Stage 4
        x = self.layer4(x)
        x = self.fusion4(x)
        x = self.attn4(x)

        # Classifier
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## Dataset and checkpoint paths


In [ ]:
CHECKPOINT_RELATIVE = 'baseline/best_model.pth'

# Works with this project's existing folder structure.
ROOT = Path.cwd().resolve()
if ROOT.name == 'reproducible':
    ROOT = ROOT.parent

candidates = [
    ROOT / 'dataset' / 'tomato_split',
    ROOT.parent / 'dataset' / 'tomato_split',
]
DATA_DIR = next(
    (p for p in candidates if (p / 'test').is_dir()),
    None
)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Dataset not found. Download PlantVillage and '
        'prepare tomato_split/train, val, test. '
        'Then update DATA_DIR in this cell.'
    )

CHECKPOINT = ROOT / CHECKPOINT_RELATIVE
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)

print('Dataset:', DATA_DIR)
print('Checkpoint:', CHECKPOINT)


## Test loader ? original preprocessing


In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])
test_dataset = datasets.ImageFolder(
    DATA_DIR / 'test', transform=transform
)
assert len(test_dataset.classes) == 10
test_loader = DataLoader(
    test_dataset, batch_size=32,
    shuffle=False, num_workers=0
)
print('Test images:', len(test_dataset))
print('Classes:', test_dataset.classes)


## Load trained model and evaluate


In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
model = HPDCNet(num_classes=10)
state = torch.load(
    CHECKPOINT, map_location='cpu', weights_only=True
)
model.load_state_dict(state)
model = model.to(device).eval()

true_labels = []
pred_labels = []
probabilities = []

with torch.inference_mode():
    for images, labels in test_loader:
        scores = model(images.to(device))
        probs = torch.softmax(scores, dim=1)
        true_labels.extend(labels.numpy())
        pred_labels.extend(probs.argmax(1).cpu().numpy())
        probabilities.extend(probs.cpu().numpy())

y = np.asarray(true_labels)
pred = np.asarray(pred_labels)
probs = np.asarray(probabilities)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y, pred, labels=np.arange(10),
        average='macro', zero_division=0
    )
)
auprc = average_precision_score(
    np.eye(10)[y], probs, average='macro'
)
print('Model:', 'Original HPDC-Net baseline')
print('Device:', device)
print('Parameters:', sum(p.numel() for p in model.parameters()))
print(f'Accuracy: {accuracy_score(y, pred):.6f}')
print(f'Macro-Precision: {precision:.6f}')
print(f'Macro-Recall: {recall:.6f}')
print(f'Macro-F1: {f1:.6f}')
print(f'Macro-AUPRC: {auprc:.6f}')
print('Confusion matrix:')
print(confusion_matrix(y, pred, labels=np.arange(10)))
